# Pipeline C: Donor Upgrade Scorer

**Object Name:** Donor Upgrade Potential Prediction

**Goal:** Identify active donors who are currently "undergiving" relative to their engagement profile and are most likely to increase their support if asked.

**Business Problem:** The development team has limited time for one-on-one calls. Instead of just chasing churn, they need to know which loyal donors are ready to be "upgraded" to a higher level of partnership.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 1. Setup Data Paths
REPO_ROOT = Path.cwd().parent.parent
DATA_DIR = REPO_ROOT / "data" / "lighthouse_csv_v7"

def load_table(name):
    fp = DATA_DIR / f"{name}.csv"
    return pd.read_csv(fp) if fp.exists() else pd.DataFrame()

donors = load_table("supporters")
donations = load_table("donations")
allocations = load_table("donation_allocations")

## Feature Engineering

We define "Upgrade" as a donor whose total giving in the 6 months following a reference point is 50% higher than their historical 6-month average.

In [ ]:
donations['donation_date'] = pd.to_datetime(donations['donation_date'])
donations['amount'] = pd.to_numeric(donations['amount'], errors='coerce')

# Reference Date (e.g., 6 months ago)
ref_date = donations['donation_date'].max() - pd.Timedelta(days=180)

# History Features
hist = donations[donations['donation_date'] < ref_date]
outcome = donations[donations['donation_date'] >= ref_date]

donor_hist = hist.groupby('supporter_id').agg({
    'amount': ['count', 'sum', 'mean', 'std'],
    'donation_date': 'max'
})
donor_hist.columns = ['gift_count', 'gift_sum', 'avg_gift', 'gift_std', 'latest_gift_date']
donor_hist['recency'] = (ref_date - donor_hist['latest_gift_date']).dt.days

# Program diversity from allocations
prog_div = allocations.merge(donations[['donation_id', 'supporter_id']], on='donation_id')
prog_div = prog_div.groupby('supporter_id')['program_area'].nunique().rename('program_diversity')

# Target: Upgrade Potential
outcome_sum = outcome.groupby('supporter_id')['amount'].sum().rename('post_ref_sum')
df = donor_hist.join(outcome_sum, how='left').fillna(0)
df = df.join(prog_div, how='left').fillna(0)

df['is_upgrade'] = (df['post_ref_sum'] >= (df['gift_sum'] / 2) * 1.5).astype(int)

# Merge with demographics
df = df.reset_index().merge(donors[['supporter_id', 'relationship_type', 'acquisition_channel']], on='supporter_id', how='left')

## Modeling

1. **Explanatory (Logistic):** Which engagement profiles are most associated with future upgrades?
2. **Predictive (Random Forest):** Rank active donors for the development team's call list.

In [ ]:
num_feats = ['gift_count', 'avg_gift', 'recency', 'program_diversity']
cat_feats = ['relationship_type', 'acquisition_channel']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_feats),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_feats)
])

lr_pipe = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(class_weight='balanced'))])
rf_pipe = Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced'))])

X = df[num_feats + cat_feats]
y = df['is_upgrade']

lr_pipe.fit(X, y)
rf_pipe.fit(X, y)

print("Explanatory AUC:", roc_auc_score(y, lr_pipe.predict_proba(X)[:, 1]))
print("Predictive AUC:", roc_auc_score(y, rf_pipe.predict_proba(X)[:, 1]))